# SmartEdu Analytics - Phase 1: Exploratory Data Analysis & Preprocessing
**Big Data-Driven Student Performance Analysis and Dropout Risk Prediction System**

### Objectives:
1. Ingestion of the official UCI Machine Learning Repository dataset: *Predict Students' Dropout and Academic Success* (UCI ID: 697).
2. Statistical distribution analysis across demographics, socioeconomics, attendance, and semester-wise academic performance.
3. Verification of ground truth Dropout labels vs. derived academic risk indicators.
4. Data cleaning, standardizing column conventions for Big Data ecosystems (Hadoop/Hive/Spark).
5. Leakage-free preprocessing, feature scaling, and stratified splitting.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 6)

# Paths relative to notebooks directory
RAW_PATH = os.path.join('..', 'dataset', 'raw', 'raw_student_data.csv')
CLEAN_PATH = os.path.join('..', 'dataset', 'processed', 'cleaned_student_data.csv')
PREP_PATH = os.path.join('..', 'dataset', 'processed', 'preprocessed_student_data.csv')

df_raw = pd.read_csv(RAW_PATH, sep=';', encoding='utf-8-sig')
print('Raw Dataset Dimensions:', df_raw.shape)

## 1. Dataset Dimensions & Schema Inspection

In [ ]:
print('First 5 Records:')
display(df_raw.head())

print('Total Missing Values (NaNs):', df_raw.isnull().sum().sum())
print('Duplicate Records:', df_raw.duplicated().sum())

## 2. Target Variable Analysis (Ground Truth Dropout Outcomes)
The dataset features three institutional status classes:
- **Graduate**: Successfully completed degree program
- **Dropout**: Discontinued before completion
- **Enrolled**: Currently enrolled in coursework

In [ ]:
target_counts = df_raw['Target'].value_counts()
print('Target Outcome Breakdown:')
print(target_counts)

fig, ax = plt.subplots(1, 2, figsize=(14, 5))
sns.barplot(x=target_counts.index, y=target_counts.values, ax=ax[0], hue=target_counts.index, palette=['#2ecc71', '#e74c3c', '#f39c12'], legend=False)
ax[0].set_title('Student Academic Outcome Counts', fontsize=13, fontweight='bold')
ax[0].set_ylabel('Number of Students')

ax[1].pie(target_counts.values, labels=target_counts.index, autopct='%1.1f%%', startangle=140, colors=['#2ecc71', '#e74c3c', '#f39c12'])
ax[1].set_title('Outcome Proportion (%)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Academic Progression: Enrolled vs Approved Units

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=df_raw,
    x='Curricular units 1st sem (enrolled)',
    y='Curricular units 1st sem (approved)',
    hue='Target',
    palette={'Graduate': '#2ecc71', 'Dropout': '#e74c3c', 'Enrolled': '#f39c12'},
    alpha=0.6,
    s=40
)
plt.title('Semester 1: Enrolled vs Approved Units by Academic Outcome', fontsize=13, fontweight='bold')
plt.plot([0, 15], [0, 15], 'k--', label='100% Approval Line')
plt.legend(title='Outcome')
plt.tight_layout()
plt.show()

## 4. Preprocessed Data & Big Data Readiness
Review of standardized snake_case schema, primary keys (`student_id`), and derived risk indicators.

In [ ]:
df_prep = pd.read_csv(PREP_PATH)
print('Preprocessed Dataset Shape:', df_prep.shape)
sample_cols = ['student_id', 'sem1_grade', 'sem2_grade', 'annual_grade_avg', 'academic_risk_score', 'academic_risk_level', 'is_dropout', 'target_outcome']
display(df_prep[sample_cols].head(10))

print('\nDerived Risk Level vs Administrative Dropout Outcome:')
print(pd.crosstab(df_prep['academic_risk_level'], df_prep['target_outcome'], margins=True))